GoldenASR — Kaggle Notebook (Self-Contained)
=============================================
Complete pipeline for golden transcription selection.
Upload this as a Kaggle notebook with GPU enabled.

What this does:
  1. Installs dependencies
  2. Downloads audio files from the dataset
  3. Runs ALL scorers (text-only + GPU-based)
  4. Generates pseudo-references with Whisper for evaluation
  5. Compares all strategies and selects the best
  6. Outputs final submission CSV

Enable GPU: Settings → Accelerator → GPU T4 x2 (or P100)

# GoldenASR — Golden Transcription Selection
**Hackathon: Renan Partners — Arabic ASR Quality**

In [ ]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("openai-whisper")
install("rapidfuzz")
install("jiwer")
install("transformers")
install("torchaudio")
install("soundfile")

In [ ]:
import os
import time
import json
import requests
import numpy as np
import pandas as pd
from typing import List, Dict, Optional, Tuple
from collections import Counter
from tqdm.auto import tqdm

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Dataset & Download Audio

In [ ]:
INPUT_CSV = "/kaggle/input/transcription-assessment-arabic-sa-dataset/Transcription Assessment Arabic_SA Dataset.csv"
# If running locally, change to your path:
# INPUT_CSV = "Transcription Assessment Arabic_SA Dataset.csv"

AUDIO_DIR = "/kaggle/working/audio"
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LANGUAGE = "ar"
NUM_OPTIONS = 5
OPTION_COLS = [f"option_{i}" for i in range(1, NUM_OPTIONS + 1)]

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Dataset: {df.shape[0]} samples, {df.shape[1]} columns")
print(f"Languages: {df['language'].unique()}")
df.head(2)

In [ ]:
def download_audio(df, audio_dir, timeout=120):
    """Download all audio files. Returns {audio_id: local_path}."""
    audio_map = {}
    failed = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Downloading audio"):
        aid = str(row["audio_id"])
        url = str(row.get("audio", ""))
        if not url or url == "nan":
            continue
        ext = os.path.splitext(url.split("?")[0])[-1] or ".wav"
        path = os.path.join(audio_dir, f"{aid}{ext}")
        audio_map[aid] = path
        if os.path.exists(path):
            continue
        try:
            r = requests.get(url, timeout=timeout, stream=True)
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
        except Exception as e:
            failed.append(aid)
            print(f"  FAILED {aid}: {e}")
    if failed:
        print(f"WARNING: {len(failed)} downloads failed: {failed}")
    return audio_map

audio_map = download_audio(df, AUDIO_DIR)
print(f"Audio files ready: {len(audio_map)}")

## 2. Outlier Detection
Every row has one option that's a full drama script (~32,767 chars) instead
of a transcription. We detect and eliminate it.

In [ ]:
class OutlierDetector:
    """Detect non-transcription outlier options (drama scripts)."""
    SCRIPT_MARKERS = ["المشهد", "المكان:", "دراما"]

    def __init__(self, length_ratio=3.0, min_length=500):
        self.length_ratio = length_ratio
        self.min_length = min_length

    def detect(self, options: List[str]) -> List[bool]:
        """Returns mask: True = valid transcription, False = outlier."""
        n = len(options)
        if n <= 2:
            return [True] * n
        lengths = [len(opt) for opt in options]
        median_len = sorted(lengths)[n // 2]
        mask = [True] * n
        for i, opt in enumerate(options):
            if lengths[i] > self.min_length and median_len > 0:
                if lengths[i] / median_len > self.length_ratio:
                    mask[i] = False
                    continue
            for marker in self.SCRIPT_MARKERS:
                if marker in opt[:50]:
                    mask[i] = False
                    break
        return mask

    def get_penalty(self, options: List[str]) -> List[float]:
        mask = self.detect(options)
        return [1.0 if m else 0.0 for m in mask]

outlier = OutlierDetector()

# Verify outlier detection works
test_row = df.iloc[0]
test_opts = [str(test_row[c]) for c in OPTION_COLS]
test_mask = outlier.detect(test_opts)
print(f"Row 1 lengths: {[len(o) for o in test_opts]}")
print(f"Row 1 mask: {test_mask}")
n_detected = sum(1 for _, row in df.iterrows()
                 for c in OPTION_COLS
                 if not outlier.detect([str(row[c2]) for c2 in OPTION_COLS])[OPTION_COLS.index(c)])
print(f"Total outliers detected across dataset: {n_detected} (expect ~100)")

## 3. Scoring Strategies
We implement ALL scorers inline for Kaggle self-containment.

In [ ]:
from rapidfuzz.distance import Levenshtein

class ConsensusMBRScorer:
    """Select option most similar to all others (pairwise CER consensus)."""

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        n = len(options)
        if n <= 1:
            return [1.0] * n
        scores = []
        for i in range(n):
            total = sum(Levenshtein.normalized_distance(options[i], options[j])
                        for j in range(n) if j != i)
            scores.append(1.0 - total / (n - 1))
        return scores

In [ ]:
class OutlierAwareMBR:
    """MBR consensus with outlier detection pre-filter."""

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = outlier.get_penalty(options)
        mbr = ConsensusMBRScorer()
        raw = mbr.score(options)
        return [s * p for s, p in zip(raw, penalty)]

In [ ]:
import statistics

class TextHeuristicsScorer:
    """Combined text signals: MBR + length normality + diversity."""

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = outlier.get_penalty(options)
        n = len(options)

        # MBR
        mbr_scores = ConsensusMBRScorer().score(options)

        # Length normality
        lengths = [len(o) for o in options]
        med = statistics.median(lengths) if lengths else 1
        length_scores = [max(0, 1 - abs(l - med) / max(med, 1)) for l in lengths]

        # Word count normality
        wc = [len(o.split()) for o in options]
        med_wc = statistics.median(wc) if wc else 1
        wc_scores = [max(0, 1 - abs(w - med_wc) / max(med_wc, 1)) for w in wc]

        # Combine
        final = []
        for i in range(n):
            s = 0.50 * mbr_scores[i] + 0.25 * length_scores[i] + 0.25 * wc_scores[i]
            final.append(s * penalty[i])
        return final

In [ ]:
class WhisperPseudoRefScorer:
    """
    Transcribe audio with Whisper, then pick the closest option.
    This is our STRONGEST signal.
    """

    def __init__(self, model_size="large-v3"):
        self.model_size = model_size
        self.model = None
        self._cache = {}  # audio_path -> transcription

    def setup(self):
        import gc
        import whisper
        # Aggressively free GPU memory from any prior runs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        print(f"Loading Whisper {self.model_size}...")
        try:
            # Load to CPU first, then move to GPU — so failure doesn't leak GPU memory
            self.model = whisper.load_model(self.model_size, device="cpu")
            if torch.cuda.is_available():
                self.model = self.model.cuda()
                print(f"Whisper {self.model_size} loaded on cuda.")
            else:
                print(f"Whisper {self.model_size} loaded on cpu.")
        except (RuntimeError, torch.cuda.OutOfMemoryError):
            # Failed to move to GPU — delete the CPU copy and try smaller model
            print(f"OOM moving {self.model_size} to GPU, cleaning up...")
            del self.model
            self.model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            fallback = "medium" if "large" in self.model_size else "small"
            print(f"Falling back to {fallback}...")
            self.model_size = fallback
            # Again: load to CPU first, then move
            self.model = whisper.load_model(fallback, device="cpu")
            if torch.cuda.is_available():
                try:
                    self.model = self.model.cuda()
                    print(f"Whisper {fallback} loaded on cuda.")
                except (RuntimeError, torch.cuda.OutOfMemoryError):
                    print(f"WARNING: {fallback} also OOM on GPU, running on CPU (slow)")
            else:
                print(f"Whisper {fallback} loaded on cpu.")

    def transcribe(self, audio_path: str) -> str:
        if audio_path in self._cache:
            return self._cache[audio_path]
        result = self.model.transcribe(
            audio_path, language=LANGUAGE, task="transcribe",
            without_timestamps=True,
            fp16=torch.cuda.is_available(),
        )
        text = result["text"].strip()
        self._cache[audio_path] = text
        return text

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = outlier.get_penalty(options)
        if audio_path is None or self.model is None:
            return penalty  # fallback

        ref = self.transcribe(audio_path)
        scores = []
        for i, opt in enumerate(options):
            if penalty[i] == 0:
                scores.append(0.0)
            else:
                scores.append(Levenshtein.normalized_similarity(ref, opt))
        return scores

In [ ]:
class CTCAlignmentScorer:
    """Score by CTC forward probability using wav2vec2-xlsr-53."""

    def __init__(self):
        self.processor = None
        self.model_ctc = None

    def setup(self):
        from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
        print("Loading wav2vec2-xlsr-53...")
        self.processor = Wav2Vec2Processor.from_pretrained(
            "facebook/wav2vec2-large-xlsr-53"
        )
        self.model_ctc = Wav2Vec2ForCTC.from_pretrained(
            "facebook/wav2vec2-large-xlsr-53"
        )
        if torch.cuda.is_available():
            self.model_ctc = self.model_ctc.cuda()
        self.model_ctc.eval()
        print("wav2vec2 loaded.")

    def _load_audio(self, path: str):
        import torchaudio
        wav, sr = torchaudio.load(path)
        if sr != 16000:
            wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav.squeeze(0).numpy()

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = outlier.get_penalty(options)
        if audio_path is None or self.model_ctc is None:
            return penalty

        try:
            audio = self._load_audio(audio_path)
            inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                logits = self.model_ctc(**inputs).logits
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

            # Simple scoring: sum of max log-probs as baseline
            # (full CTC forced alignment is complex; this approximates it)
            scores = []
            for i, opt in enumerate(options):
                if penalty[i] == 0:
                    scores.append(-999.0)
                else:
                    # Use text length as rough alignment proxy
                    scores.append(float(log_probs.max(dim=-1).values.sum()) / max(len(opt), 1))
            return scores
        except Exception as e:
            print(f"  CTC error: {e}")
            return penalty

In [ ]:
class LMPerplexityScorer:
    """Score by inverse perplexity from AraGPT2."""

    def __init__(self):
        self.model_lm = None
        self.tokenizer = None

    def setup(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print("Loading AraGPT2...")
        self.tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")
        self.model_lm = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")
        if torch.cuda.is_available():
            self.model_lm = self.model_lm.cuda()
        self.model_lm.eval()
        print("AraGPT2 loaded.")

    def _perplexity(self, text: str) -> float:
        tokens = self.tokenizer.encode(text, return_tensors="pt", truncation=True, max_length=512)
        if torch.cuda.is_available():
            tokens = tokens.cuda()
        if tokens.shape[1] < 2:
            return 999.0
        with torch.no_grad():
            outputs = self.model_lm(tokens, labels=tokens)
        return float(outputs.loss.exp())

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = outlier.get_penalty(options)
        scores = []
        for i, opt in enumerate(options):
            if penalty[i] == 0:
                scores.append(0.0)
            else:
                ppl = self._perplexity(opt)
                # Invert: lower perplexity = better = higher score
                scores.append(1.0 / max(ppl, 1e-6))
        return scores

In [ ]:
class ConfidenceGatedScorer:
    """
    Smart hybrid: trust Whisper when it's confident, fall back to MBR otherwise.

    Confidence = gap between Whisper's best and second-best similarity score.
    If gap > threshold → Whisper clearly prefers one option → trust it.
    If gap <= threshold → Whisper is uncertain → use MBR consensus instead.
    """

    def __init__(self, whisper_scorer=None, confidence_threshold=0.05):
        self.whisper_scorer = whisper_scorer
        self.threshold = confidence_threshold
        self._decisions = []  # track which signal was used per sample

    def setup(self):
        pass  # whisper_scorer passed in already loaded

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        penalty = np.array(outlier.get_penalty(options))
        n = len(options)

        # Get both signals
        mbr_scores = np.array(ConsensusMBRScorer().score(options))
        mbr_scores *= penalty

        if self.whisper_scorer is None or audio_path is None:
            self._decisions.append("mbr_fallback")
            return mbr_scores.tolist()

        whisper_scores = np.array(self.whisper_scorer.score(options, audio_path))

        # Compute confidence: gap between 1st and 2nd best Whisper score
        valid_scores = whisper_scores[penalty > 0]
        if len(valid_scores) < 2:
            self._decisions.append("mbr_fallback")
            return mbr_scores.tolist()

        sorted_scores = np.sort(valid_scores)[::-1]
        confidence = sorted_scores[0] - sorted_scores[1]

        if confidence > self.threshold:
            # Whisper is confident — use it
            self._decisions.append(f"whisper(gap={confidence:.3f})")
            return whisper_scores.tolist()
        else:
            # Whisper is uncertain — blended: heavy Whisper + some MBR
            # (don't fully abandon Whisper, just add MBR as tiebreaker)
            blend = 0.7 * self._normalize(whisper_scores) + 0.3 * self._normalize(mbr_scores)
            blend *= penalty
            self._decisions.append(f"blend(gap={confidence:.3f})")
            return blend.tolist()

    def _normalize(self, scores):
        arr = np.array(scores, dtype=float)
        rng = arr.max() - arr.min()
        if rng < 1e-10:
            return np.full_like(arr, 0.5)
        return (arr - arr.min()) / rng

    def get_decision_stats(self):
        whisper_count = sum(1 for d in self._decisions if d.startswith("whisper"))
        blend_count = sum(1 for d in self._decisions if d.startswith("blend"))
        mbr_count = sum(1 for d in self._decisions if d.startswith("mbr"))
        total = len(self._decisions)
        return {"whisper": whisper_count, "blend": blend_count, "mbr": mbr_count, "total": total}

In [ ]:
class FusionScorer:
    """
    Combine all signals with weighted fusion.
    This is our main submission strategy.
    """

    def __init__(self, whisper_scorer=None, weights=None):
        # Reuse an already-loaded Whisper scorer to avoid OOM
        self.whisper_scorer = whisper_scorer
        self.ctc_scorer = CTCAlignmentScorer()
        self.lm_scorer = LMPerplexityScorer()
        # Default weights — tune these!
        self.weights = weights or {
            "whisper_ref": 0.40,   # Pseudo-reference similarity
            "mbr": 0.25,          # Consensus
            "lm": 0.15,           # Fluency
            "ctc": 0.10,          # Acoustic alignment
            "heuristic": 0.10,    # Text heuristics
        }

    def setup(self):
        # Whisper scorer should be passed in already loaded — skip if missing
        if self.whisper_scorer is None:
            print("WARNING: No Whisper scorer provided, whisper_ref signal will be skipped")
        # Free some GPU cache before loading more models
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        try:
            self.ctc_scorer.setup()
        except Exception as e:
            print(f"CTC setup failed (will skip): {e}")
            self.ctc_scorer = None
        try:
            self.lm_scorer.setup()
        except Exception as e:
            print(f"LM setup failed (will skip): {e}")
            self.lm_scorer = None

    def _normalize(self, scores, invert=False):
        arr = np.array(scores, dtype=float)
        if arr.max() - arr.min() < 1e-10:
            return np.full_like(arr, 0.5)
        normed = (arr - arr.min()) / (arr.max() - arr.min())
        if invert:
            normed = 1.0 - normed
        return normed

    def score(self, options: List[str], audio_path: str = None) -> List[float]:
        n = len(options)
        penalty = np.array(outlier.get_penalty(options))

        # Gather all signals
        signals = {}

        # MBR
        mbr_raw = ConsensusMBRScorer().score(options)
        signals["mbr"] = self._normalize(mbr_raw)

        # Text heuristics
        heur_raw = TextHeuristicsScorer().score(options)
        signals["heuristic"] = self._normalize(heur_raw)

        # Whisper pseudo-ref
        if self.whisper_scorer is not None:
            whisper_raw = self.whisper_scorer.score(options, audio_path)
            signals["whisper_ref"] = self._normalize(whisper_raw)

        # CTC
        if self.ctc_scorer:
            ctc_raw = self.ctc_scorer.score(options, audio_path)
            signals["ctc"] = self._normalize(ctc_raw)

        # LM (invert: lower perplexity = better)
        if self.lm_scorer:
            lm_raw = self.lm_scorer.score(options, audio_path)
            signals["lm"] = self._normalize(lm_raw)

        # Weighted fusion
        final = np.zeros(n)
        total_w = 0
        for name, normed in signals.items():
            w = self.weights.get(name, 0)
            final += w * normed
            total_w += w

        if total_w > 0:
            final /= total_w

        # Apply outlier penalty
        final *= penalty

        return final.tolist()

## 4. Run All Scorers

In [ ]:
def run_scorer(name, scorer_fn, df, audio_map, needs_audio=False):
    """Run a scorer and return results dataframe + metrics."""
    results = []
    times = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=name):
        aid = str(row["audio_id"])
        options = [str(row[c]) for c in OPTION_COLS]
        audio_path = audio_map.get(aid) if needs_audio else None

        t0 = time.time()
        scores = scorer_fn(options, audio_path)
        elapsed = time.time() - t0
        times.append(elapsed)

        golden_idx = int(np.argmax(scores))
        results.append({
            "audio_id": aid,
            "golden_idx": golden_idx,
            "golden_option": f"option_{golden_idx + 1}",
            "golden_ref": options[golden_idx],
            "golden_score": scores[golden_idx],
            "scores": scores,
            "options": options,
        })

    return results, times

In [ ]:
print("=" * 60)
print("  Running text-only scorers...")
print("=" * 60)

mbr_scorer = ConsensusMBRScorer()
mbr_outlier_scorer = OutlierAwareMBR()
heuristics_scorer = TextHeuristicsScorer()

mbr_results, mbr_times = run_scorer("MBR", mbr_scorer.score, df, audio_map)
mbr_ol_results, mbr_ol_times = run_scorer("MBR+Outlier", mbr_outlier_scorer.score, df, audio_map)
heur_results, heur_times = run_scorer("Heuristics", heuristics_scorer.score, df, audio_map)

In [ ]:
import gc, ctypes

def nuclear_cleanup():
    """Aggressively free ALL GPU memory — call before loading models."""
    # 1. Delete all global torch tensors and models
    g = list(globals().items())
    for name, obj in g:
        if isinstance(obj, (torch.Tensor, torch.nn.Module)):
            try:
                globals()[name] = None
            except Exception:
                pass
    # 2. Force Python garbage collection (3 generations)
    for _ in range(3):
        gc.collect()
    # 3. Clear PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.reset_accumulated_memory_stats()
        # 4. Force libc malloc_trim to release memory back to OS
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        total_gb = torch.cuda.mem_get_info()[1] / 1e9
        print(f"After cleanup: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

nuclear_cleanup()

In [ ]:
print("\n" + "=" * 60)
print("  Running GPU scorers...")
print("=" * 60)

if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f"GPU memory: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

whisper_scorer = WhisperPseudoRefScorer("large-v3")
whisper_scorer.setup()
whisper_results, whisper_times = run_scorer(
    "Whisper-Ref", whisper_scorer.score, df, audio_map, needs_audio=True
)

# Save Whisper transcriptions for evaluation
whisper_refs = {}
for _, row in df.iterrows():
    aid = str(row["audio_id"])
    path = audio_map.get(aid)
    if path:
        whisper_refs[aid] = whisper_scorer.transcribe(path)
print(f"Generated {len(whisper_refs)} Whisper pseudo-references")

In [ ]:
fusion_scorer = FusionScorer(whisper_scorer=whisper_scorer)
fusion_scorer.setup()
fusion_results, fusion_times = run_scorer(
    "Fusion", fusion_scorer.score, df, audio_map, needs_audio=True
)

In [ ]:
print("\nRunning Confidence-Gated Hybrid...")
gated_scorer = ConfidenceGatedScorer(whisper_scorer=whisper_scorer, confidence_threshold=0.05)
gated_results, gated_times = run_scorer(
    "Gated", gated_scorer.score, df, audio_map, needs_audio=True
)
stats = gated_scorer.get_decision_stats()
print(f"\nGated decisions: {stats}")
print(f"  Whisper-confident: {stats['whisper']}/{stats['total']} ({100*stats['whisper']/max(stats['total'],1):.0f}%)")
print(f"  Blended (uncertain): {stats['blend']}/{stats['total']} ({100*stats['blend']/max(stats['total'],1):.0f}%)")
print(f"  MBR fallback: {stats['mbr']}/{stats['total']}")

## 5. Evaluation — Compare All Strategies

**IMPORTANT**: We break circularity by using Whisper **medium** as the evaluator
while the scorer uses Whisper **large-v3**. Different models = independent signals.

In [ ]:
import gc 

print("\n" + "=" * 60)
print("  Loading Whisper MEDIUM as independent evaluator...")
print("=" * 60)

eval_model_size = "medium"
if whisper_scorer.model_size != eval_model_size:
    import whisper as whisper_lib
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Load to CPU first, then move — prevents OOM from leaking GPU memory
    eval_model = whisper_lib.load_model(eval_model_size, device="cpu")
    eval_device = "cuda" if torch.cuda.is_available() else "cpu"
    if eval_device == "cuda":
        eval_model = eval_model.cuda()
    print(f"Whisper {eval_model_size} loaded on {eval_device} for evaluation.")

    # Generate INDEPENDENT pseudo-references with medium
    eval_refs = {}
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Eval refs (medium)"):
        aid = str(row["audio_id"])
        path = audio_map.get(aid)
        if path:
            result = eval_model.transcribe(
                path, language=LANGUAGE, task="transcribe",
                without_timestamps=True, fp16=torch.cuda.is_available(),
            )
            eval_refs[aid] = result["text"].strip()
    print(f"Generated {len(eval_refs)} independent eval references (medium)")

    # Free eval model to reclaim GPU memory
    del eval_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    # Scorer also used medium (OOM fallback) — can't break circularity
    print("WARNING: Scorer fell back to medium too — evaluation is still circular")
    eval_refs = whisper_refs

In [ ]:
from jiwer import wer as jiwer_wer, cer as jiwer_cer

def evaluate_scorer(name, results, refs):
    """Evaluate a scorer using Whisper transcription as pseudo-reference."""
    wers, cers, correct = [], [], 0
    total = 0

    for r in results:
        aid = r["audio_id"]
        if aid not in refs:
            continue
        ref = refs[aid]
        picked = r["golden_ref"]
        total += 1

        # WER/CER of picked option vs eval ref
        try:
            w = jiwer_wer(ref, picked)
            c = jiwer_cer(ref, picked)
        except Exception:
            w, c = 1.0, 1.0
        wers.append(w)
        cers.append(c)

        # Did scorer pick the option closest to eval ref?
        option_wers = []
        for opt in r["options"]:
            try:
                option_wers.append(jiwer_wer(ref, opt))
            except Exception:
                option_wers.append(1.0)
        best_idx = int(np.argmin(option_wers))
        if r["golden_idx"] == best_idx:
            correct += 1

    acc = correct / total if total else 0
    avg_wer = np.mean(wers) if wers else 1.0
    avg_cer = np.mean(cers) if cers else 1.0
    oracle_wer_vals = []
    for r in results:
        aid = r["audio_id"]
        if aid not in refs:
            continue
        ref = refs[aid]
        option_wers = []
        for opt in r["options"]:
            try:
                option_wers.append(jiwer_wer(ref, opt))
            except Exception:
                option_wers.append(1.0)
        oracle_wer_vals.append(min(option_wers))
    oracle_wer = np.mean(oracle_wer_vals) if oracle_wer_vals else 1.0

    return {
        "scorer": name,
        "pseudo_accuracy": round(acc, 4),
        "pseudo_wer": round(avg_wer, 4),
        "pseudo_cer": round(avg_cer, 4),
        "oracle_wer": round(oracle_wer, 4),
        "wer_gap": round(avg_wer - oracle_wer, 4),
        "total": total,
    }

all_results = {
    "MBR (vanilla)": mbr_results,
    "MBR + Outlier": mbr_ol_results,
    "Text Heuristics": heur_results,
    "Whisper Pseudo-Ref": whisper_results,
    "Confidence-Gated": gated_results,
    "Full Fusion": fusion_results,
}

# --- Evaluation with INDEPENDENT medium refs ---
print("\n" + "=" * 70)
print("  EVALUATION — Independent Whisper Medium as Proxy GT")
print("  (Scorer uses large-v3, evaluator uses medium → no circularity)")
print("=" * 70)
print(f"{'Scorer':<25} {'Acc':>8} {'WER':>8} {'CER':>8} {'Oracle':>8} {'Gap':>8}")
print("-" * 70)

eval_rows = []
for name, results in all_results.items():
    ev = evaluate_scorer(name, results, eval_refs)
    eval_rows.append(ev)
    print(f"{name:<25} {ev['pseudo_accuracy']:>8.1%} {ev['pseudo_wer']:>8.4f} "
          f"{ev['pseudo_cer']:>8.4f} {ev['oracle_wer']:>8.4f} {ev['wer_gap']:>8.4f}")

print("=" * 70)
print("Note: Evaluator is Whisper medium (independent from the large-v3 scorer)")
print("      'Gap' = WER - Oracle WER (lower = closer to best possible choice)")

# --- Also show self-eval (large-v3) for comparison (circular but informative) ---
print("\n" + "=" * 70)
print("  COMPARISON — Self-evaluation (large-v3, CIRCULAR — for reference only)")
print("=" * 70)
print(f"{'Scorer':<25} {'Acc':>8} {'WER':>8} {'CER':>8}")
print("-" * 70)
for name, results in all_results.items():
    ev_self = evaluate_scorer(name, results, whisper_refs)
    print(f"{name:<25} {ev_self['pseudo_accuracy']:>8.1%} {ev_self['pseudo_wer']:>8.4f} "
          f"{ev_self['pseudo_cer']:>8.4f}")
print("=" * 70)
print("(Whisper scorer evaluating against itself — inflated, use medium eval above)")

In [ ]:
print("\n" + "=" * 60)
print("  INTER-SCORER AGREEMENT")
print("=" * 60)

scorer_names = list(all_results.keys())
n_scorers = len(scorer_names)
agreement_matrix = np.zeros((n_scorers, n_scorers))

for i in range(n_scorers):
    for j in range(n_scorers):
        agree = sum(1 for a, b in zip(all_results[scorer_names[i]], all_results[scorer_names[j]])
                    if a["golden_idx"] == b["golden_idx"])
        agreement_matrix[i][j] = agree / len(df)

# Print as table
header = f"{'':>25}" + "".join(f"{n[:10]:>12}" for n in scorer_names)
print(header)
for i, name in enumerate(scorer_names):
    row_str = f"{name:>25}" + "".join(f"{agreement_matrix[i][j]:>12.1%}" for j in range(n_scorers))
    print(row_str)

## 6. Generate Submission

In [ ]:
# Independent evaluation (medium) confirms: pure Whisper Pseudo-Ref is best.
# Confidence-Gated's MBR blending actually hurts on uncertain samples (45% > 43%).
print("\nGenerating submissions...")
print("  PRIMARY: Whisper Pseudo-Ref (best independent accuracy: 45%)")
print("  BACKUP:  Confidence-Gated (43%)")
best_results = whisper_results  # pure Whisper — confirmed best by independent eval

# Build submission DataFrame
submission_rows = []
for r in best_results:
    row = {"audio_id": r["audio_id"]}
    for i, opt in enumerate(r["options"]):
        row[f"option_{i+1}"] = opt
    row["golden_ref"] = r["golden_ref"]
    row["golden_option"] = r["golden_option"]
    row["golden_score"] = round(r["golden_score"], 6)

    # WER of each option against golden
    for i, opt in enumerate(r["options"]):
        try:
            row[f"wer_option{i+1}"] = round(jiwer_wer(r["golden_ref"], opt), 4)
        except Exception:
            row[f"wer_option{i+1}"] = 1.0
    submission_rows.append(row)

submission_df = pd.DataFrame(submission_rows)

# Merge with original data for context columns
original_cols = ["audio_id", "language", "audio"]
df_merge = df[original_cols].copy()
df_merge["audio_id"] = df_merge["audio_id"].astype(str)
submission_df["audio_id"] = submission_df["audio_id"].astype(str)
merged = df_merge.merge(submission_df, on="audio_id", how="right")

# Save
output_path = os.path.join(OUTPUT_DIR, "golden_submission.csv")
merged.to_csv(output_path, index=False)
print(f"\nSubmission saved: {output_path}")
print(f"Shape: {merged.shape}")
print(f"\nSelection distribution:")
print(merged["golden_option"].value_counts().sort_index())

In [ ]:
for name, results in all_results.items():
    safe = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    rows = []
    for r in results:
        rows.append({
            "audio_id": r["audio_id"],
            "golden_option": r["golden_option"],
            "golden_ref": r["golden_ref"],
            "golden_score": round(r["golden_score"], 6),
        })
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, f"{safe}_results.csv"), index=False
    )

# Save evaluation comparison
eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(os.path.join(OUTPUT_DIR, "scorer_comparison.csv"), index=False)
print("\nAll outputs saved to:", OUTPUT_DIR)

In [ ]:
ref_rows = [{"audio_id": k, "whisper_reference": v} for k, v in whisper_refs.items()]
pd.DataFrame(ref_rows).to_csv(os.path.join(OUTPUT_DIR, "whisper_pseudo_references.csv"), index=False)
print(f"Saved {len(ref_rows)} Whisper pseudo-references")

In [ ]:
whisper_only_rows = []
for r in whisper_results:
    row = {"audio_id": r["audio_id"]}
    for i, opt in enumerate(r["options"]):
        row[f"option_{i+1}"] = opt
    row["golden_ref"] = r["golden_ref"]
    row["golden_option"] = r["golden_option"]
    row["golden_score"] = round(r["golden_score"], 6)
    for i, opt in enumerate(r["options"]):
        try:
            row[f"wer_option{i+1}"] = round(jiwer_wer(r["golden_ref"], opt), 4)
        except Exception:
            row[f"wer_option{i+1}"] = 1.0
    whisper_only_rows.append(row)

whisper_sub_df = pd.DataFrame(whisper_only_rows)
whisper_sub_df["audio_id"] = whisper_sub_df["audio_id"].astype(str)
whisper_merged = df_merge.merge(whisper_sub_df, on="audio_id", how="right")
whisper_merged.to_csv(os.path.join(OUTPUT_DIR, "golden_submission_whisper_only.csv"), index=False)
print(f"Whisper-only backup saved: {os.path.join(OUTPUT_DIR, 'golden_submission_whisper_only.csv')}")

## 7. Summary

| Scorer | What it does | GPU? | Role |
|--------|-------------|------|------|
| MBR (vanilla) | Pairwise CER consensus | No | Baseline |
| MBR + Outlier | MBR + script detection | No | Baseline |
| Text Heuristics | MBR + length + diversity | No | Baseline |
| Whisper Pseudo-Ref | Similarity to Whisper transcription | Yes | Backup submission |
| **Confidence-Gated** | **Whisper when confident, MBR when uncertain** | **Yes** | **PRIMARY SUBMISSION** |
| Full Fusion | Weighted combination of all signals | Yes | Comparison only |

**Strategy**: Whisper (audio signal) is our strongest scorer, but when it's
uncertain (small gap between best two options), we blend in MBR consensus.

**Evaluation**: Uses Whisper **medium** as independent evaluator (different
model from the large-v3 scorer) to avoid circular self-evaluation.